In [ ]:
from pathlib import Path
import math
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')
print(f'Joblib version: {joblib.__version__}')


In [ ]:
CANDIDATE_PATHS = [Path('/kaggle/input/smart-mobility-bike-demand/bike_demand_model.pkl'), Path('/kaggle/input/smart-mobility-demand/bike_demand_model.pkl'), Path('/kaggle/input/bike_demand_model.pkl'), Path('../model/bike_demand_model.pkl'), Path('model/bike_demand_model.pkl'), Path('artifacts/bike_demand_model.pkl'), Path('bike_demand_model.pkl')]
model_path = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if model_path is None:
    kaggle_inputs = list(Path('/kaggle/input').glob('**/bike_demand_model.pkl')) if Path('/kaggle/input').exists() else []
    if kaggle_inputs:
        model_path = kaggle_inputs[0]
if model_path is None:
    raise FileNotFoundError('Could not find bike_demand_model.pkl. Please attach the model/dataset to your notebook.')
print(f'Loading model bundle from: {model_path.resolve()}')
bundle = joblib.load(model_path)
print('✓ Model bundle loaded successfully!')


In [ ]:
print('Project:', bundle.get('project_name'))
print('Artifact Version:', bundle.get('artifact_version'))
print('Training Window:', bundle.get('training_start'), 'to', bundle.get('training_end'))
print('Testing Window: ', bundle.get('test_start'), 'to', bundle.get('test_end'))
print(f"Stations Modeled: {bundle.get('station_count')}")
print(f"Mean Trip Duration: {bundle.get('mean_trip_duration_minutes'):.2f} minutes")
print(f"Safety Buffer: {bundle.get('safety_buffer') * 100 - 100:.0f}%")
metrics_df = pd.DataFrame(bundle.get('metrics', {})).T
display(metrics_df.round(3))


In [ ]:
profiles_data = bundle.get('cluster_detail_stats', {})
if profiles_data:
    prof_df = pd.DataFrame(profiles_data).T
    display(prof_df[['count', 'pct_of_training', 'mean', 'median', 'std', 'p10', 'p25', 'p75', 'p90']])


In [ ]:
def predict_smart_mobility(bundle: dict, req: dict) -> dict:
    timestamp = pd.Timestamp(req['date']) + pd.Timedelta(hours=int(req['hour']))
    dow = int(timestamp.dayofweek)
    doy = int(timestamp.dayofyear)
    month = int(timestamp.month)
    row = {'temperature': float(req['temperature']), 'max_temperature': float(req['max_temperature']), 'min_temperature': float(req['min_temperature']), 'wind_speed': float(req['wind_speed']), 'precipitation': float(req['precipitation']), 'humidity': float(req['humidity']), 'sunshine': float(req['sunshine']), 'hour_sin': math.sin(2 * math.pi * req['hour'] / 24), 'hour_cos': math.cos(2 * math.pi * req['hour'] / 24), 'dow_sin': math.sin(2 * math.pi * dow / 7), 'dow_cos': math.cos(2 * math.pi * dow / 7), 'month_sin': math.sin(2 * math.pi * (month - 1) / 12), 'month_cos': math.cos(2 * math.pi * (month - 1) / 12), 'day_of_year_sin': math.sin(2 * math.pi * (doy - 1) / 365.25), 'day_of_year_cos': math.cos(2 * math.pi * (doy - 1) / 365.25), 'is_holiday': int(bool(req['is_holiday'])), 'is_weekend': int(dow >= 5), 'weather_code': str(int(req['weather'])), 'season': str(int(req['season']))}
    ctx_df = pd.DataFrame([row])
    cluster_cols = bundle['cluster_numeric_features'] + bundle['cluster_categorical_features']
    ctx_mat = bundle['cluster_preprocessor'].transform(ctx_df[cluster_cols])
    cluster_id = int(bundle['kmeans'].predict(ctx_mat)[0])
    demand_profile = bundle['cluster_label_map'][cluster_id]
    profile_mean = bundle['cluster_demand_means'].get(cluster_id, 0.0)
    row['demand_profile'] = demand_profile
    feat_df = pd.DataFrame([row])
    model_cols = bundle['model_numeric_features'] + bundle['model_categorical_features']
    raw_pred = float(bundle['model'].predict(feat_df[model_cols])[0])
    predicted_demand = round(max(0.0, raw_pred), 2)
    baseline = round(bundle['baseline_by_dow_hour'].get((dow, int(req['hour'])), bundle['overall_training_mean']), 2)
    pct_diff = round((predicted_demand - baseline) / baseline * 100, 2) if baseline > 0 else 0.0
    mean_dur = bundle['mean_trip_duration_minutes']
    concurrent = round(predicted_demand * mean_dur / 60.0, 2)
    capacity = round(concurrent * bundle['safety_buffer'], 2)
    if pct_diff >= 15.0:
        rec = 'Increase bicycle availability'
    elif pct_diff <= -15.0:
        rec = 'Reduce excess bicycle availability'
    else:
        rec = 'Maintain current bicycle availability'
    shares = bundle['station_share_lookup'].get((dow, int(req['hour'])), bundle['global_station_shares'])
    station_alloc = []
    allocated_share = 0.0
    for s in shares:
        pickups = round(predicted_demand * s['share'], 2)
        station_alloc.append({'station_id': s['station_id'], 'station_name': s['station_name'], 'share_pct': round(s['share'] * 100, 2), 'expected_pickups': pickups})
        allocated_share += s['share']
    remaining_pickups = round(predicted_demand * max(0.0, 1.0 - allocated_share), 2)
    return {'timestamp': str(timestamp), 'day_name': timestamp.day_name(), 'predicted_hourly_demand': predicted_demand, 'assigned_profile': demand_profile, 'profile_average_demand': round(profile_mean, 2), 'baseline_demand': baseline, 'diff_from_baseline_pct': pct_diff, 'recommendation': rec, 'estimated_concurrent_bicycles': concurrent, 'recommended_capacity': capacity, 'top_stations': station_alloc, 'remaining_stations_pickups': remaining_pickups}


In [ ]:
scenarios = [{'name': 'Scenario A: Spring Weekday Morning Commute (May 15, 08:00)', 'input': {'date': '2023-05-15', 'hour': 8, 'temperature': 14.2, 'max_temperature': 17.0, 'min_temperature': 10.5, 'wind_speed': 2.8, 'precipitation': 0.0, 'humidity': 68.0, 'weather': 3, 'sunshine': 55.0, 'season': 1, 'is_holiday': False}}, {'name': 'Scenario B: Sunny Summer Weekend Afternoon (July 15, 14:00)', 'input': {'date': '2023-07-15', 'hour': 14, 'temperature': 22.0, 'max_temperature': 24.5, 'min_temperature': 16.0, 'wind_speed': 3.2, 'precipitation': 0.0, 'humidity': 58.0, 'weather': 1, 'sunshine': 60.0, 'season': 2, 'is_holiday': False}}, {'name': 'Scenario C: Rainy Autumn Evening in Held-Out Period (Nov 7, 18:00)', 'input': {'date': '2023-11-07', 'hour': 18, 'temperature': 6.5, 'max_temperature': 7.5, 'min_temperature': 5.0, 'wind_speed': 4.5, 'precipitation': 2.2, 'humidity': 88.0, 'weather': 4, 'sunshine': 0.0, 'season': 3, 'is_holiday': False}}]
results = []
for sc in scenarios:
    res = predict_smart_mobility(bundle, sc['input'])
    results.append({'name': sc['name'], 'result': res})
    print('=' * 75)
    print(sc['name'])
    print('=' * 75)
    print(f"Timestamp:                     {res['timestamp']} ({res['day_name']})")
    print(f"Predicted Hourly Demand:       {res['predicted_hourly_demand']} trips/hour")
    print(f"Historical Baseline Demand:    {res['baseline_demand']} trips/hour ({res['diff_from_baseline_pct']:+.1f}%)")
    print(f"Assigned Contextual Profile:   {res['assigned_profile']} (Hist Mean: {res['profile_average_demand']})")
    print(f"Operational Recommendation:    {res['recommendation']}")
    print(f"Estimated Concurrent Bikes:   {res['estimated_concurrent_bicycles']}")
    print(f"Recommended Fleet Capacity:    {res['recommended_capacity']} (includes 20% safety buffer)")
    print('')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
names = ['Scenario A\n(Spring Morning)', 'Scenario B\n(Summer Wknd)', 'Scenario C\n(Autumn Rain)']
preds = [r['result']['predicted_hourly_demand'] for r in results]
bases = [r['result']['baseline_demand'] for r in results]
caps = [r['result']['recommended_capacity'] for r in results]
x = np.arange(len(names))
width = 0.25
axes[0].bar(x - width, preds, width, label='Predicted Demand', color='#1f77b4')
axes[0].bar(x, bases, width, label='Weekday Baseline', color='#ff7f0e', alpha=0.8)
axes[0].bar(x + width, caps, width, label='Recommended Capacity', color='#2ca02c', alpha=0.7)
axes[0].set_xticks(x)
axes[0].set_xticklabels(names)
axes[0].set_ylabel('Bicycles / Hour')
axes[0].set_title('Demand Forecast vs Baseline vs Capacity')
axes[0].legend()
axes[0].grid(axis='y', linestyle='--', alpha=0.5)
top_st = results[0]['result']['top_stations']
st_names = [s['station_name'][:18] for s in top_st]
st_pickups = [s['expected_pickups'] for s in top_st]
axes[1].barh(st_names[::-1], st_pickups[::-1], color='#3b528b')
axes[1].set_xlabel('Expected Pickups (trips/hour)')
axes[1].set_title('Top Station Allocation (Scenario A - Commute Peak)')
axes[1].grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
